# Educational breast-cancer classification example

**Non-clinical:** this notebook is not a medical device, diagnostic tool, or clinically validated model. Never use its outputs for individual or clinical decisions. The repository does not establish the dataset provenance or `diagnosis` class meanings; verify both independently. No dataset is distributed.

This workflow reserves disjoint stratified training, validation, and test partitions. It trains only on the training partition, reports aggregate validation metrics, then evaluates the held-out test partition once. Do not use test results to tune model choices. It prints no individual rows or paired predictions.


In [ ]:
import json
import os
from pathlib import Path

from breast_cancer_model import (
    evaluate_model,
    fit_model,
    load_dataset,
    save_model_artifact,
    split_dataset,
)


## Load and validate authorized local data

By default, the CSV is resolved from the notebook's current working directory. Set `BREAST_CANCER_DATA` to a different authorized local CSV path. Keep private data outside the repository and do not commit it. Required feature and target columns are documented in the README.


In [ ]:
data_path = Path(os.environ.get("BREAST_CANCER_DATA", "Breast_cancer_data.csv")).expanduser()
X, y = load_dataset(data_path)
print(f"Validated {len(X)} rows and {len(X.columns)} configured input features.")


## Create disjoint data partitions

The split is deterministic and stratified. Validation data can support future model selection. The test set is reserved for final evaluation; reviewing its result and then changing the model compromises that evaluation. Small datasets may not have enough examples per class for all three partitions.


In [ ]:
splits = split_dataset(X, y, validation_size=0.15, test_size=0.20, random_state=42)
print(
    "Partition sizes:",
    {
        "train": len(splits.y_train),
        "validation": len(splits.y_validation),
        "test": len(splits.y_test),
    },
)


## Fit the fixed educational model

The estimator is a Random Forest. No scaler is used because scaling is not required for this estimator. Only training rows are passed to `fit`. The fixed settings are an example, not a tuned or clinically selected model.


In [ ]:
model = fit_model(splits.X_train, splits.y_train, n_estimators=200, random_state=42)


## Validation metrics

These are dataset- and split-specific aggregate metrics. They do not establish external validity, clinical performance, or class meaning.


In [ ]:
validation_metrics = evaluate_model(model, splits.X_validation, splits.y_validation)
print(json.dumps(validation_metrics, indent=2))


## Final held-out test evaluation

Evaluate this partition only after modeling choices are fixed. If choices are changed after reviewing these results, do not treat the same test results as an unbiased final evaluation.


In [ ]:
test_metrics = evaluate_model(model, splits.X_test, splits.y_test)
print(json.dumps(test_metrics, indent=2))


## Save a local model artifact

The artifact includes the input schema and runtime package versions and is written atomically with mode 0600 on POSIX; Windows permissions follow filesystem ACLs. It is ignored by Git. Set `BREAST_CANCER_ARTIFACT` to change the local output path. **Only load artifacts you created or fully trust:** joblib/pickle deserialization can execute code. The artifact is not a clinical or secure serving format.


In [ ]:
artifact_path = Path(
    os.environ.get("BREAST_CANCER_ARTIFACT", "models/breast_cancer_model.joblib")
).expanduser()
save_model_artifact(model, artifact_path)
print("Saved the local model artifact.")
